# Policy-to-Logic RL Environment — Training Notebook

This notebook runs the **reward-guided trajectory optimization loop** against the deployed environment.

**What it does:**
1. Connects to the live HF Spaces environment
2. Runs 8 episodes per task (3 tasks = 24 total episodes)
3. Accumulates high-reward trajectories as few-shot examples
4. Generates training evidence plots (reward curve, accuracy curve, improvement chart)

In [ ]:
# Cell 1: Install dependencies
!pip install openai requests matplotlib numpy

In [ ]:
# Cell 2: Configuration
import os

# SET THESE BEFORE RUNNING
HF_TOKEN = ""  # Your Hugging Face token with inference access
ENV_URL = "https://godreign-policy2logic.hf.space"  # Your deployed environment URL

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["ENV_BASE_URL"] = ENV_URL

print(f"Environment URL: {ENV_URL}")
print(f"HF Token set: {'Yes' if HF_TOKEN else 'NO - MUST SET THIS'}")

In [ ]:
# Cell 3: Verify environment is reachable
import requests

r = requests.get(f"{ENV_URL}/health")
print(f"Status: {r.status_code}")
print(f"Response: {r.json()}")

r2 = requests.get(f"{ENV_URL}/tasks")
tasks = r2.json()
print(f"\nAvailable tasks: {list(tasks['tasks'].keys())}")

In [ ]:
# Cell 4: Training loop implementation
# Full contents of training/trajectory_optimizer.py

import json
import os
import time
import requests
from dataclasses import dataclass, field
from typing import Optional
from openai import OpenAI

# ── Configuration ────────────────────────────────────────────────────────────

ENV_BASE_URL = os.getenv("ENV_BASE_URL", "http://localhost:7860")
HF_TOKEN = os.getenv("HF_TOKEN", "")
MODEL = "Qwen/Qwen2.5-72B-Instruct"
TEMPERATURE = 0.3
MAX_TOKENS = 1024

NUM_EPISODES_PER_TASK = 8
TOP_K_TRAJECTORIES = 3
MIN_REWARD_THRESHOLD = 0.3
TASKS = ["data_access", "resource_access", "transaction_approval"]

@dataclass
class Step:
    step_number: int
    action_type: str
    action_content: str
    reward: float
    accuracy: float
    feedback: str
    clarification_response: Optional[str] = None

@dataclass
class Trajectory:
    task_name: str
    episode_id: int
    steps: list[Step] = field(default_factory=list)
    total_reward: float = 0.0
    final_accuracy: float = 0.0
    success: bool = False

    def to_few_shot_string(self) -> str:
        lines = [f"=== Example Episode (reward={self.total_reward:.2f}, accuracy={self.final_accuracy:.2f}) ==="]
        for s in self.steps:
            lines.append(f"Step {s.step_number}: action={s.action_type}")
            lines.append(f"  Content: {s.action_content[:200]}")
            lines.append(f"  Result: accuracy={s.accuracy:.2f}, reward={s.reward:.2f}")
            if s.feedback:
                lines.append(f"  Feedback: {s.feedback[:150]}")
        return "\n".join(lines)

class EnvClient:
    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()

    def reset(self, task_name: str) -> dict:
        r = self.session.post(f"{self.base_url}/reset", json={"task_name": task_name})
        r.raise_for_status()
        return r.json()

    def step(self, action_type: str, content: str) -> dict:
        r = self.session.post(f"{self.base_url}/step", json={"action_type": action_type, "content": content})
        r.raise_for_status()
        return r.json()

    def health(self) -> bool:
        try:
            r = self.session.get(f"{self.base_url}/health", timeout=5)
            return r.status_code == 200
        except Exception:
            return False

class Agent:
    def __init__(self, hf_token: str):
        self.client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)

    def get_action(self, observation, step_number, episode_history, few_shot_examples):
        system_prompt = self._build_system_prompt(few_shot_examples)
        user_prompt = self._build_user_prompt(observation, step_number, episode_history)
        try:
            response = self.client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                temperature=TEMPERATURE, max_tokens=MAX_TOKENS
            )
            raw = response.choices[0].message.content.strip()
            return self._parse_response(raw, observation)
        except Exception as e:
            print(f"    [LLM ERROR] {e}")
            return "propose_rules", json.dumps({"rules": [], "default": "DENY"})

    def _build_system_prompt(self, few_shot_examples):
        base = """You are a policy-to-logic agent. Convert natural language policies into executable rules.

AVAILABLE ACTIONS:
1. ask_clarification: {"type": "clarification", "question": "your question"}
2. propose_rules: {"rules": [...], "default": "DECISION"}
3. refine_rules: {"rules": [...], "default": "DECISION"}

DSL FORMAT: {"rules": [{"if": [{"field": "NAME", "op": "OP", "value": VAL}], "then": "DECISION"}], "default": "FALLBACK"}
Operators: >, <, >=, <=, ==, !=. Rules execute top-to-bottom, first match wins.

STRATEGY: Ask 1-2 clarifications first, then propose rules, then refine based on failures.
OUTPUT: Respond ONLY with valid JSON: {"action_type": "...", "content": "..."}"""
        if few_shot_examples:
            base += "\n\nLEARNED FROM PREVIOUS EPISODES:\n"
            for traj in few_shot_examples[-TOP_K_TRAJECTORIES:]:
                base += "\n" + traj.to_few_shot_string() + "\n"
        return base

    def _build_user_prompt(self, obs, step, history):
        lines = [f"TASK: {obs.get('task_name', 'unknown')}", f"STEP: {step} of {obs.get('max_steps', 7)}", f"\nPOLICY:\n{obs.get('policy_text', '')}"]
        if obs.get("clarification_response"): lines.append(f"\nLAST CLARIFICATION:\n{obs['clarification_response']}")
        if obs.get("test_results"):
            tr = obs["test_results"]
            lines.append(f"\nTEST RESULTS: {tr.get('passed',0)}/{tr.get('total',0)} (acc={obs.get('current_accuracy',0):.2f})")
            if tr.get("sample_failures"): lines.extend([f"  - {f}" for f in tr["sample_failures"][:3]])
        if obs.get("feedback"): lines.append(f"\nFEEDBACK: {obs['feedback']}")
        if history: lines.append(f"\nHISTORY:\n" + "\n".join(history[-3:]))
        lines.append(f"\nAVAILABLE: {obs.get('available_actions', [])}")
        lines.append("\nRespond with JSON only.")
        return "\n".join(lines)

    def _parse_response(self, raw, obs):
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"): raw = raw[4:]
        raw = raw.strip()
        try:
            parsed = json.loads(raw)
            action_type = parsed.get("action_type", "propose_rules")
            content = parsed.get("content", "{}")
            valid = obs.get("available_actions", ["propose_rules"])
            if action_type not in valid: action_type = valid[0]
            if isinstance(content, dict): content = json.dumps(content)
            return action_type, content
        except: return "propose_rules", json.dumps({"rules": [], "default": "DENY"})

class TrajectoryBank:
    def __init__(self): self.bank = {task: [] for task in TASKS}
    def store(self, t):
        if t.total_reward >= MIN_REWARD_THRESHOLD:
            self.bank[t.task_name].append(t)
            self.bank[t.task_name].sort(key=lambda x: x.total_reward, reverse=True)
            self.bank[t.task_name] = self.bank[t.task_name][:TOP_K_TRAJECTORIES]
    def get_examples(self, task): return self.bank.get(task, [])
    def summary(self): return {t: {"stored": len(v), "best_reward": max((x.total_reward for x in v), default=0)} for t,v in self.bank.items()}

class TrainingLoop:
    def __init__(self, env_url, hf_token):
        self.env = EnvClient(env_url)
        self.agent = Agent(hf_token)
        self.bank = TrajectoryBank()
        self.metrics = []

    def run_episode(self, task_name, episode_id):
        few_shots = self.bank.get_examples(task_name)
        traj = Trajectory(task_name=task_name, episode_id=episode_id)
        result = self.env.reset(task_name)
        obs, done, history = result.get("observation", {}), result.get("done", False), []
        print(f"  [Episode {episode_id}] task={task_name} few_shots={len(few_shots)}")
        step_num = 0
        while not done and step_num < obs.get("max_steps", 7):
            step_num += 1
            action_type, content = self.agent.get_action(obs, step_num, history, few_shots)
            result = self.env.step(action_type, content)
            reward, done = result.get("reward", 0.0), result.get("done", False)
            obs, info = result.get("observation", {}), result.get("info", {})
            step = Step(step_num, action_type, content[:300], reward, obs.get("current_accuracy", 0.0), obs.get("feedback", "") or "", obs.get("clarification_response"))
            traj.steps.append(step); traj.total_reward += reward
            history.append(f"Step {step_num}: {action_type} -> reward={reward:.2f} acc={step.accuracy:.2f}")
            print(f"    step={step_num} action={action_type} reward={reward:.3f} acc={step.accuracy:.2f}")
            if done:
                traj.final_accuracy = info.get("episode_score", obs.get("current_accuracy", 0.0))
                traj.success = obs.get("current_accuracy", 0.0) >= 0.9
                break
        if not traj.steps: traj.final_accuracy = 0.0
        return traj

    def run(self):
        print("=" * 60)
        print("REWARD-GUIDED TRAJECTORY OPTIMIZATION")
        print(f"Tasks: {TASKS}, Episodes/task: {NUM_EPISODES_PER_TASK}")
        print("=" * 60)
        if not self.env.health(): raise RuntimeError(f"Env not reachable at {ENV_BASE_URL}")
        print(f"Environment: OK\n")
        global_ep = 0
        for task in TASKS:
            print(f"\n--- TASK: {task} ---")
            task_rewards = []
            for ep in range(1, NUM_EPISODES_PER_TASK + 1):
                global_ep += 1
                traj = self.run_episode(task, ep)
                self.bank.store(traj)
                self.metrics.append({"global_episode": global_ep, "task": task, "episode_in_task": ep, "total_reward": traj.total_reward, "final_accuracy": traj.final_accuracy, "success": traj.success, "num_steps": len(traj.steps)})
                task_rewards.append(traj.total_reward)
                print(f"  -> Ep {ep}: reward={traj.total_reward:.3f} acc={traj.final_accuracy:.2f} success={traj.success}")
                time.sleep(0.5)
            print(f"  Improvement: {task_rewards[-1] - task_rewards[0]:+.3f}")
        print("\n" + "=" * 60 + "\nTRAINING COMPLETE\n" + "=" * 60)
        return self.metrics

def save_plots(metrics):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt; import numpy as np
    os.makedirs("training/plots", exist_ok=True)
    episodes = [m["global_episode"] for m in metrics]
    rewards = [m["total_reward"] for m in metrics]
    colors = {"data_access": "#2196F3", "resource_access": "#FF9800", "transaction_approval": "#4CAF50"}
    # Plot 1: Reward
    fig, ax = plt.subplots(figsize=(10, 5))
    for task in TASKS:
        te = [m["global_episode"] for m in metrics if m["task"]==task]
        tr = [m["total_reward"] for m in metrics if m["task"]==task]
        ax.plot(te, tr, marker="o", label=task, color=colors.get(task), linewidth=2, markersize=5)
    z = np.polyfit(episodes, rewards, 1); p = np.poly1d(z)
    ax.plot(episodes, p(episodes), "--", color="red", alpha=0.5, label="trend")
    ax.set_xlabel("Episode"); ax.set_ylabel("Total Reward"); ax.set_title("Reward Curve"); ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)
    plt.tight_layout(); plt.savefig("training/plots/reward_curve.png", dpi=150); plt.close()
    # Plot 2: Accuracy
    fig, ax = plt.subplots(figsize=(10, 5))
    for task in TASKS:
        te = [m["global_episode"] for m in metrics if m["task"]==task]
        ta = [m["final_accuracy"] for m in metrics if m["task"]==task]
        ax.plot(te, ta, marker="s", label=task, color=colors.get(task), linewidth=2, markersize=5)
    ax.axhline(y=0.9, color="red", linestyle="--", alpha=0.7, label="threshold")
    ax.set_xlabel("Episode"); ax.set_ylabel("Accuracy"); ax.set_title("Accuracy Curve"); ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)
    plt.tight_layout(); plt.savefig("training/plots/accuracy_curve.png", dpi=150); plt.close()
    # Plot 3: Improvement
    fig, ax = plt.subplots(figsize=(8, 5))
    tnames, imps = [], []
    for task in TASKS:
        accs = [m["final_accuracy"] for m in metrics if m["task"]==task]
        if len(accs) >= 2: tnames.append(task.replace("_","\n")); imps.append(accs[-1]-accs[0])
    bars = ax.bar(tnames, imps, color=["#2196F3","#FF9800","#4CAF50"][:len(tnames)])
    ax.axhline(y=0, color="black"); ax.set_ylabel("Improvement"); ax.set_title("Per-Task Improvement"); ax.grid(True, axis="y", alpha=0.3)
    for bar, val in zip(bars, imps): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{val:+.2f}", ha="center", fontweight="bold")
    plt.tight_layout(); plt.savefig("training/plots/improvement_chart.png", dpi=150); plt.close()
    with open("training/plots/metrics.json", "w") as f: json.dump(metrics, f, indent=2)
    print("All plots saved to training/plots/")

In [ ]:
# Cell 5: Run training loop
loop = TrainingLoop(ENV_URL, HF_TOKEN)
metrics = loop.run()
print(f"\nTotal episodes run: {len(metrics)}")

In [ ]:
# Cell 6: Generate plots and display inline
save_plots(metrics)

from IPython.display import Image, display
display(Image("training/plots/reward_curve.png"))
display(Image("training/plots/accuracy_curve.png"))
display(Image("training/plots/improvement_chart.png"))

In [ ]:
# Cell 7: Download plots to commit to repo
# After running this, download the files and commit them to your GitHub repo
from google.colab import files

files.download("training/plots/reward_curve.png")
files.download("training/plots/accuracy_curve.png")
files.download("training/plots/improvement_chart.png")
files.download("training/plots/metrics.json")

print("Downloaded. Now commit these files to: training/plots/ in your repo.")